In [ ]:
# STEP 0 - Personal parameters (keep this as the first cell)
SID4 = 6441
SEED = 6441
SLICE = 441
HP_ID = 3
CLS_A = 1
CLS_B = 7

print("Personal parameters")
print({
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
})
print("STEP 0: PASS")


# DATA 266 - Homework 6
## Self-Supervised and Contrastive Representation Learning on STL-10

**Student:** Nikhil Kanaparthi  
**SID4:** 6441

This notebook completes all four assignment parts with the same unpretrained ResNet-18 backbone:

1. Supervised training with exactly 500 labeled images.
2. Rotation self-supervised pretraining on all 100,000 unlabeled images, followed by a frozen linear probe.
3. SimCLR-style contrastive pretraining on 20,000 unlabeled images, followed by a frozen linear probe.
4. Top-5 cosine nearest-neighbor visualization for three identical query images across all encoders.

### Before running

- In Colab choose **Runtime -> Change runtime type -> GPU**.
- An A100/L4 is preferred; a T4 works but is slower.
- Keep `FAST_DEV_RUN = False` for the final submission.
- Google Drive checkpointing is enabled so training can resume after a disconnect.
- Run the notebook from top to bottom. Each completed epoch is checkpointed.


## 1. Imports, reproducibility, GPU check, and persistent folders


In [ ]:
import gc
import json
import math
import os
import platform
import random
import shutil
import sys
import time
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import STL10
from torchvision.models import resnet18

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime -> Change runtime type -> GPU, then rerun."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3

USE_GOOGLE_DRIVE = True
IN_COLAB = Path("/content").exists()

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/DATA266_HW6_6441")
    DATA_ROOT = Path("/content/stl10_data")
elif IN_COLAB:
    WORK_ROOT = Path("/content/DATA266_HW6_6441")
    DATA_ROOT = Path("/content/stl10_data")
else:
    WORK_ROOT = Path.cwd() / "DATA266_HW6_6441"
    DATA_ROOT = WORK_ROOT / "data"

OUTPUT_DIR = WORK_ROOT / "HW6_STL10_SSL_Nikhil_Kanaparthi_6441"
CHECKPOINT_DIR = WORK_ROOT / "checkpoints"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

for directory in [WORK_ROOT, DATA_ROOT, OUTPUT_DIR, CHECKPOINT_DIR, FIGURE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOG_LINES = []

def log(message):
    line = f"[{datetime.now(timezone.utc).isoformat()}] {message}"
    print(line)
    RUN_LOG_LINES.append(line)

log(f"Student=Nikhil Kanaparthi SID4={SID4} SEED={SEED} SLICE={SLICE} HP_ID={HP_ID}")
log(f"Python={platform.python_version()} PyTorch={torch.__version__} Torchvision={torchvision.__version__}")
log(f"GPU={GPU_NAME} GPU_memory_GiB={GPU_MEMORY_GIB:.2f}")

print("Device:", DEVICE)
print("GPU:", GPU_NAME)
print(f"GPU memory: {GPU_MEMORY_GIB:.2f} GiB")
print("Persistent work folder:", WORK_ROOT)
print("STEP 1: PASS")


## 2. Assignment configuration


In [ ]:
# Keep False for the final submitted run. True is only for a quick code smoke test.
FAST_DEV_RUN = False

IMAGE_SIZE = 96
NUM_CLASSES = 10
EMBEDDING_DIM = 512
LABELED_SUBSET_SIZE = 500
SIMCLR_UNLABELED_SIZE = 20_000
TEMPERATURE = 0.2

SUPERVISED_EPOCHS = 15
ROTATION_EPOCHS = 15
ROTATION_LINEAR_EPOCHS = 20
SIMCLR_EPOCHS = 15
SIMCLR_LINEAR_EPOCHS = 20

if FAST_DEV_RUN:
    LABELED_SUBSET_SIZE = 100
    SIMCLR_UNLABELED_SIZE = 512
    SUPERVISED_EPOCHS = 1
    ROTATION_EPOCHS = 1
    ROTATION_LINEAR_EPOCHS = 1
    SIMCLR_EPOCHS = 1
    SIMCLR_LINEAR_EPOCHS = 1

SUPERVISED_BATCH_SIZE = 64
SSL_BATCH_SIZE = 256 if GPU_MEMORY_GIB >= 30 else 128
EVAL_BATCH_SIZE = 256
NUM_WORKERS = 2
USE_AMP = DEVICE.type == "cuda"

print("Final-run mode:", not FAST_DEV_RUN)
print("Supervised epochs:", SUPERVISED_EPOCHS)
print("Rotation SSL epochs:", ROTATION_EPOCHS)
print("Rotation linear-probe epochs:", ROTATION_LINEAR_EPOCHS)
print("SimCLR epochs:", SIMCLR_EPOCHS)
print("SimCLR linear-probe epochs:", SIMCLR_LINEAR_EPOCHS)
print("SSL batch size:", SSL_BATCH_SIZE)
print("Temperature:", TEMPERATURE)

if FAST_DEV_RUN:
    print("WARNING: FAST_DEV_RUN is only a smoke test and must not be submitted.")
else:
    assert LABELED_SUBSET_SIZE == 500
    assert SIMCLR_UNLABELED_SIZE == 20_000
assert TEMPERATURE == 0.2

log(
    f"Configuration image_size={IMAGE_SIZE} labeled_subset={LABELED_SUBSET_SIZE} "
    f"simclr_subset={SIMCLR_UNLABELED_SIZE} tau={TEMPERATURE} "
    f"epochs=({SUPERVISED_EPOCHS},{ROTATION_EPOCHS},{ROTATION_LINEAR_EPOCHS},"
    f"{SIMCLR_EPOCHS},{SIMCLR_LINEAR_EPOCHS})"
)
print("STEP 2: PASS")


## 3. Load STL-10 and create the balanced 10% labeled subset

STL-10 contains 5,000 labeled training images, 8,000 test images, and 100,000 unlabeled images. The assignment's 10% subset is constructed as exactly 50 examples from each of the 10 classes, giving 500 labeled images total. The same indices are reused for Part A and both linear probes.


In [ ]:
STL10_MEAN = (0.4467, 0.4398, 0.4066)
STL10_STD = (0.2603, 0.2566, 0.2713)

train_raw = STL10(DATA_ROOT, split="train", download=True, transform=None)
test_raw = STL10(DATA_ROOT, split="test", download=True, transform=None)
unlabeled_raw = STL10(DATA_ROOT, split="unlabeled", download=True, transform=None)

CLASS_NAMES = list(train_raw.classes)
train_labels = np.asarray(train_raw.labels)
test_labels = np.asarray(test_raw.labels)

def balanced_subset_indices(labels, total_size, seed):
    labels = np.asarray(labels)
    classes = np.unique(labels)
    if total_size % len(classes) != 0:
        raise ValueError("Subset size must divide evenly across classes.")
    per_class = total_size // len(classes)
    rng = np.random.default_rng(seed)
    selected = []
    for class_id in classes:
        candidates = np.flatnonzero(labels == class_id)
        selected.extend(rng.choice(candidates, size=per_class, replace=False).tolist())
    rng.shuffle(selected)
    return selected

labeled_indices = balanced_subset_indices(train_labels, LABELED_SUBSET_SIZE, SEED)

subset_counts = pd.Series(train_labels[labeled_indices]).value_counts().sort_index()
subset_table = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "selected_images": [int(subset_counts.get(i, 0)) for i in range(NUM_CLASSES)],
})
display(subset_table)

assert len(labeled_indices) == 500
assert subset_table["selected_images"].eq(50).all()

print("Labeled training images:", len(train_raw))
print("Selected labeled subset:", len(labeled_indices))
print("Unlabeled images:", len(unlabeled_raw))
print("Test images:", len(test_raw))

log(
    f"STL10 train={len(train_raw)} unlabeled={len(unlabeled_raw)} test={len(test_raw)} "
    f"balanced_labeled_subset={len(labeled_indices)}"
)
print("STEP 3: PASS")


## 4. Transforms, reusable dataset wrappers, and data loaders


In [ ]:
supervised_train_transform = T.Compose([
    T.RandomCrop(IMAGE_SIZE, padding=12),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(STL10_MEAN, STL10_STD),
])

evaluation_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(STL10_MEAN, STL10_STD),
])

rotation_transform = T.Compose([
    T.RandomCrop(IMAGE_SIZE, padding=12),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(STL10_MEAN, STL10_STD),
])

# The four required demo augmentations are:
# 1) RandomResizedCrop, 2) RandomHorizontalFlip,
# 3) ColorJitter, and 4) RandomGrayscale.
simclr_transform = T.Compose([
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(STL10_MEAN, STL10_STD),
])

class IndexedTransformDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, label = self.base_dataset[self.indices[position]]
        return self.transform(image), int(label)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)

labeled_train_dataset = IndexedTransformDataset(
    train_raw, labeled_indices, supervised_train_transform
)
test_dataset = IndexedTransformDataset(
    test_raw, range(len(test_raw)), evaluation_transform
)

labeled_train_loader = DataLoader(
    labeled_train_dataset,
    batch_size=SUPERVISED_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

def denormalize_image(tensor):
    image = tensor.detach().cpu().clone()
    mean = torch.tensor(STL10_MEAN).view(3, 1, 1)
    std = torch.tensor(STL10_STD).view(3, 1, 1)
    return (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

batch_images, batch_labels = next(iter(labeled_train_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for axis, image, label in zip(axes.flat, batch_images[:10], batch_labels[:10]):
    axis.imshow(denormalize_image(image))
    axis.set_title(CLASS_NAMES[int(label)])
    axis.axis("off")
fig.suptitle("Balanced 500-image labeled subset: augmented training examples")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "labeled_subset_examples.png", dpi=180, bbox_inches="tight")
plt.show()

print("Training batch:", batch_images.shape, batch_labels.shape)
print("STEP 4: PASS")


## 5. Shared ResNet-18 backbone and training utilities

Every method creates a separate standard ResNet-18 with `weights=None`. The 1,000-class ImageNet head is replaced by an identity layer, producing a 512-dimensional embedding. No pretrained weights are used.


In [ ]:
class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.output_dim = EMBEDDING_DIM

    def forward(self, images):
        return self.backbone(images)

class EncoderClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.output_dim, num_classes)

    def forward(self, images):
        return self.classifier(self.encoder(images))

class ProjectionHead(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, hidden_dim=512, output_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, embeddings):
        return self.network(embeddings)

def parameter_counts(module):
    total = sum(parameter.numel() for parameter in module.parameters())
    trainable = sum(
        parameter.numel() for parameter in module.parameters() if parameter.requires_grad
    )
    return total, trainable

def freeze_encoder(encoder):
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    encoder.eval()

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

@torch.no_grad()
def evaluate_classifier(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)
    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }

def save_training_checkpoint(path, epoch, model, optimizer, scheduler, scaler, history):
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict(),
        "history": history,
        "seed": SEED,
    }
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)

def load_training_checkpoint(path, model, optimizer, scheduler, scaler):
    if not path.exists():
        return 0, []
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    if scheduler is not None and checkpoint.get("scheduler_state") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state"])
    if checkpoint.get("scaler_state"):
        scaler.load_state_dict(checkpoint["scaler_state"])
    completed_epoch = int(checkpoint["epoch"])
    history = list(checkpoint.get("history", []))
    print(f"Resuming from {path.name}: completed epoch {completed_epoch}")
    return completed_epoch, history

def train_classifier(
    model,
    train_loader,
    epochs,
    learning_rate,
    checkpoint_path,
    run_name,
    head_only=False,
):
    criterion = nn.CrossEntropyLoss()
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        trainable_parameters,
        lr=learning_rate,
        momentum=0.9,
        weight_decay=5e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = make_scaler()
    start_epoch, history = load_training_checkpoint(
        checkpoint_path, model, optimizer, scheduler, scaler
    )

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train()
        if head_only:
            model.encoder.eval()
            model.classifier.train()

        epoch_loss = 0.0
        epoch_correct = 0
        epoch_examples = 0
        start_time = time.time()

        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item() * labels.size(0)
            epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
            epoch_examples += labels.size(0)

        scheduler.step()
        row = {
            "stage": run_name,
            "epoch": epoch,
            "loss": epoch_loss / epoch_examples,
            "accuracy": epoch_correct / epoch_examples,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - start_time,
        }
        history.append(row)
        print(
            f"[{run_name} {epoch:02d}/{epochs}] "
            f"loss={row['loss']:.4f} accuracy={100*row['accuracy']:.2f}% "
            f"time={row['seconds']:.1f}s"
        )
        log(json.dumps(row, sort_keys=True))
        save_training_checkpoint(
            checkpoint_path, epoch, model, optimizer, scheduler, scaler, history
        )

    return history

print("ResNet-18 and training utilities ready.")
print("STEP 5: PASS")


# Part A - Supervised Learning with Limited Labels

The first ResNet-18 is trained end-to-end from random initialization using only the balanced 500-image subset for 15 epochs. Test data is used only for the final evaluation.


In [ ]:
supervised_encoder = ResNet18Encoder()
supervised_model = EncoderClassifier(supervised_encoder, NUM_CLASSES).to(DEVICE)

supervised_total, supervised_trainable = parameter_counts(supervised_model)
print(f"Part A total parameters: {supervised_total:,}")
print(f"Part A trainable parameters: {supervised_trainable:,}")

supervised_history = train_classifier(
    model=supervised_model,
    train_loader=labeled_train_loader,
    epochs=SUPERVISED_EPOCHS,
    learning_rate=0.05,
    checkpoint_path=CHECKPOINT_DIR / "part_a_supervised_latest.pt",
    run_name="Part A supervised",
    head_only=False,
)

part_a_test = evaluate_classifier(supervised_model, test_loader)
print(f"Part A test loss: {part_a_test['loss']:.4f}")
print(f"Part A test accuracy: {100*part_a_test['accuracy']:.2f}%")
log(f"Part A test_metrics={json.dumps(part_a_test, sort_keys=True)}")
print("PART A: PASS")


# Part B - Rotation Self-Supervised Learning

Each of all 100,000 unlabeled images is randomly rotated by 0, 90, 180, or 270 degrees. A separate unpretrained ResNet-18 and four-class rotation head are trained for 15 epochs. The rotation head is then removed, the encoder is frozen, and only a new linear 10-class layer is trained on the same 500 labeled examples.


In [ ]:
class RotationDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, _ = self.base_dataset[index]
        rotation_label = random.randint(0, 3)
        rotated = T.functional.rotate(image, 90 * rotation_label)
        return self.transform(rotated), rotation_label

rotation_source = (
    Subset(unlabeled_raw, range(512)) if FAST_DEV_RUN else unlabeled_raw
)
rotation_dataset = RotationDataset(rotation_source, rotation_transform)
rotation_loader = DataLoader(
    rotation_dataset,
    batch_size=SSL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED + 1),
)

sample_image, _ = unlabeled_raw[SLICE % len(unlabeled_raw)]
rotation_degrees = [0, 90, 180, 270]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for axis, degrees in zip(axes, rotation_degrees):
    axis.imshow(T.functional.rotate(sample_image, degrees))
    axis.set_title(f"Rotation label {degrees // 90}: {degrees} degrees")
    axis.axis("off")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "rotation_pretext_examples.png", dpi=180, bbox_inches="tight")
plt.show()

print("Rotation dataset size:", len(rotation_dataset))
if not FAST_DEV_RUN:
    assert len(rotation_dataset) == 100_000
print("PART B DATA: PASS")


In [ ]:
def train_rotation_model(model, loader, epochs, checkpoint_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = make_scaler()
    start_epoch, history = load_training_checkpoint(
        checkpoint_path, model, optimizer, scheduler, scaler
    )

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_examples = 0
        start_time = time.time()

        for images, rotation_labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            rotation_labels = rotation_labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(images)
                loss = criterion(logits, rotation_labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item() * rotation_labels.size(0)
            epoch_correct += (logits.argmax(dim=1) == rotation_labels).sum().item()
            epoch_examples += rotation_labels.size(0)

        scheduler.step()
        row = {
            "stage": "Part B rotation SSL",
            "epoch": epoch,
            "loss": epoch_loss / epoch_examples,
            "accuracy": epoch_correct / epoch_examples,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - start_time,
        }
        history.append(row)
        print(
            f"[Rotation SSL {epoch:02d}/{epochs}] "
            f"loss={row['loss']:.4f} rotation_accuracy={100*row['accuracy']:.2f}% "
            f"time={row['seconds']:.1f}s"
        )
        log(json.dumps(row, sort_keys=True))
        save_training_checkpoint(
            checkpoint_path, epoch, model, optimizer, scheduler, scaler, history
        )

    return history

rotation_encoder = ResNet18Encoder()
rotation_model = EncoderClassifier(rotation_encoder, 4).to(DEVICE)
rotation_total, rotation_trainable = parameter_counts(rotation_model)
print(f"Rotation model total/trainable: {rotation_total:,} / {rotation_trainable:,}")

rotation_history = train_rotation_model(
    rotation_model,
    rotation_loader,
    ROTATION_EPOCHS,
    CHECKPOINT_DIR / "part_b_rotation_ssl_latest.pt",
)

print("Final rotation loss:", f"{rotation_history[-1]['loss']:.4f}")
print("Final rotation accuracy:", f"{100*rotation_history[-1]['accuracy']:.2f}%")
print("PART B PRETRAINING: PASS")


In [ ]:
# Remove the rotation head by reusing only rotation_model.encoder.
rotation_encoder = rotation_model.encoder
freeze_encoder(rotation_encoder)
rotation_linear_model = EncoderClassifier(rotation_encoder, NUM_CLASSES).to(DEVICE)

rotation_encoder_trainable = sum(
    parameter.numel() for parameter in rotation_linear_model.encoder.parameters()
    if parameter.requires_grad
)
rotation_head_trainable = sum(
    parameter.numel() for parameter in rotation_linear_model.classifier.parameters()
    if parameter.requires_grad
)

print("Frozen encoder trainable parameters:", rotation_encoder_trainable)
print("Linear head trainable parameters:", rotation_head_trainable)
assert rotation_encoder_trainable == 0
assert rotation_head_trainable > 0

rotation_linear_history = train_classifier(
    model=rotation_linear_model,
    train_loader=labeled_train_loader,
    epochs=ROTATION_LINEAR_EPOCHS,
    learning_rate=0.1,
    checkpoint_path=CHECKPOINT_DIR / "part_b_rotation_linear_latest.pt",
    run_name="Part B rotation linear probe",
    head_only=True,
)

part_b_test = evaluate_classifier(rotation_linear_model, test_loader)
print(f"Part B test loss: {part_b_test['loss']:.4f}")
print(f"Part B test accuracy: {100*part_b_test['accuracy']:.2f}%")
log(f"Part B test_metrics={json.dumps(part_b_test, sort_keys=True)}")
print("PART B: PASS")


# Part C - SimCLR-Style Contrastive Learning

The SimCLR subset contains 20,000 seeded unlabeled images. Each image produces two independent views using the demo's four augmentations: random resized crop, horizontal flip, color jitter, and random grayscale. NT-Xent uses cosine similarity and temperature 0.2. After 15 epochs, the projection head is removed and the frozen encoder is evaluated with a 20-epoch linear probe on the same 500 labels.


In [ ]:
class TwoViewsDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, _ = self.base_dataset[self.indices[position]]
        return self.transform(image), self.transform(image)

simclr_rng = np.random.default_rng(SEED)
simclr_indices = simclr_rng.choice(
    len(unlabeled_raw), size=SIMCLR_UNLABELED_SIZE, replace=False
).tolist()

simclr_dataset = TwoViewsDataset(unlabeled_raw, simclr_indices, simclr_transform)
simclr_loader = DataLoader(
    simclr_dataset,
    batch_size=SSL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED + 2),
)

view1, view2 = simclr_dataset[0]
original_image, _ = unlabeled_raw[simclr_indices[0]]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(original_image)
axes[0].set_title("Original")
axes[1].imshow(denormalize_image(view1))
axes[1].set_title("Augmented view 1")
axes[2].imshow(denormalize_image(view2))
axes[2].set_title("Augmented view 2")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "simclr_two_views.png", dpi=180, bbox_inches="tight")
plt.show()

print("SimCLR subset size:", len(simclr_dataset))
if not FAST_DEV_RUN:
    assert len(simclr_dataset) == 20_000
print("PART C DATA: PASS")


In [ ]:
def nt_xent_loss(projected_view1, projected_view2, temperature=0.2):
    # Float32 similarity computation improves numerical stability under AMP.
    projected_view1 = F.normalize(projected_view1.float(), dim=1)
    projected_view2 = F.normalize(projected_view2.float(), dim=1)
    batch_size = projected_view1.size(0)

    representations = torch.cat([projected_view1, projected_view2], dim=0)
    similarities = representations @ representations.T / temperature

    diagonal_mask = torch.eye(
        2 * batch_size, device=similarities.device, dtype=torch.bool
    )
    similarities = similarities.masked_fill(diagonal_mask, float("-inf"))

    positives = torch.cat([
        torch.diag(similarities, batch_size),
        torch.diag(similarities, -batch_size),
    ])
    denominator = torch.logsumexp(similarities, dim=1)
    return (-positives + denominator).mean()

class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ResNet18Encoder()
        self.projection_head = ProjectionHead()

    def forward(self, images):
        embeddings = self.encoder(images)
        return self.projection_head(embeddings)

def train_simclr(model, loader, epochs, checkpoint_path):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = make_scaler()
    start_epoch, history = load_training_checkpoint(
        checkpoint_path, model, optimizer, scheduler, scaler
    )

    for epoch in range(start_epoch + 1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_batches = 0
        start_time = time.time()

        for view1, view2 in loader:
            view1 = view1.to(DEVICE, non_blocking=True)
            view2 = view2.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                projection1 = model(view1)
                projection2 = model(view2)
                loss = nt_xent_loss(projection1, projection2, TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            epoch_batches += 1

        scheduler.step()
        row = {
            "stage": "Part C SimCLR",
            "epoch": epoch,
            "loss": epoch_loss / epoch_batches,
            "accuracy": np.nan,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - start_time,
        }
        history.append(row)
        print(
            f"[SimCLR {epoch:02d}/{epochs}] "
            f"NT-Xent loss={row['loss']:.4f} time={row['seconds']:.1f}s"
        )
        log(json.dumps(row, sort_keys=True, default=str))
        save_training_checkpoint(
            checkpoint_path, epoch, model, optimizer, scheduler, scaler, history
        )

    return history

simclr_model = SimCLRModel().to(DEVICE)
simclr_total, simclr_trainable = parameter_counts(simclr_model)
print(f"SimCLR total/trainable: {simclr_total:,} / {simclr_trainable:,}")

with torch.no_grad():
    preview_view1, preview_view2 = next(iter(simclr_loader))
    preview_z1 = simclr_model(preview_view1[:8].to(DEVICE))
    preview_z2 = simclr_model(preview_view2[:8].to(DEVICE))
    preview_loss = nt_xent_loss(preview_z1, preview_z2, TEMPERATURE)
print("Projection shape:", preview_z1.shape)
print("Initial preview NT-Xent loss:", float(preview_loss))

simclr_history = train_simclr(
    simclr_model,
    simclr_loader,
    SIMCLR_EPOCHS,
    CHECKPOINT_DIR / "part_c_simclr_latest.pt",
)

print("Final SimCLR NT-Xent loss:", f"{simclr_history[-1]['loss']:.4f}")
print("PART C PRETRAINING: PASS")


In [ ]:
# Remove the projection head by reusing only simclr_model.encoder.
simclr_encoder = simclr_model.encoder
freeze_encoder(simclr_encoder)
simclr_linear_model = EncoderClassifier(simclr_encoder, NUM_CLASSES).to(DEVICE)

simclr_encoder_trainable = sum(
    parameter.numel() for parameter in simclr_linear_model.encoder.parameters()
    if parameter.requires_grad
)
simclr_head_trainable = sum(
    parameter.numel() for parameter in simclr_linear_model.classifier.parameters()
    if parameter.requires_grad
)

print("Frozen encoder trainable parameters:", simclr_encoder_trainable)
print("Linear head trainable parameters:", simclr_head_trainable)
assert simclr_encoder_trainable == 0
assert simclr_head_trainable > 0

simclr_linear_history = train_classifier(
    model=simclr_linear_model,
    train_loader=labeled_train_loader,
    epochs=SIMCLR_LINEAR_EPOCHS,
    learning_rate=0.1,
    checkpoint_path=CHECKPOINT_DIR / "part_c_simclr_linear_latest.pt",
    run_name="Part C SimCLR linear probe",
    head_only=True,
)

part_c_test = evaluate_classifier(simclr_linear_model, test_loader)
print(f"Part C test loss: {part_c_test['loss']:.4f}")
print(f"Part C test accuracy: {100*part_c_test['accuracy']:.2f}%")
log(f"Part C test_metrics={json.dumps(part_c_test, sort_keys=True)}")
print("PART C: PASS")


# Part D - Cosine Nearest-Neighbor Visualization

Three test queries with different class labels are selected deterministically. The exact same query indices are used for the supervised, rotation-SSL, and SimCLR encoders. Embeddings are L2-normalized, so their dot product equals cosine similarity. Each query is excluded from its own neighbor list.


In [ ]:
@torch.no_grad()
def compute_embeddings(encoder, loader):
    encoder.eval()
    embeddings = []
    labels = []
    for images, batch_labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast_context():
            batch_embeddings = encoder(images)
        embeddings.append(F.normalize(batch_embeddings.float(), dim=1).cpu())
        labels.append(batch_labels.cpu())
    return torch.cat(embeddings), torch.cat(labels)

def choose_distinct_class_queries(labels, number_of_queries, seed):
    rng = np.random.default_rng(seed)
    selected = []
    seen_classes = set()
    for index in rng.permutation(len(labels)):
        label = int(labels[index])
        if label not in seen_classes:
            selected.append(int(index))
            seen_classes.add(label)
        if len(selected) == number_of_queries:
            break
    return selected

supervised_embeddings, embedding_labels = compute_embeddings(
    supervised_model.encoder, test_loader
)
rotation_embeddings, rotation_test_labels = compute_embeddings(
    rotation_linear_model.encoder, test_loader
)
simclr_embeddings, simclr_test_labels = compute_embeddings(
    simclr_linear_model.encoder, test_loader
)

assert torch.equal(embedding_labels, rotation_test_labels)
assert torch.equal(embedding_labels, simclr_test_labels)

QUERY_INDICES = choose_distinct_class_queries(embedding_labels.numpy(), 3, SEED)
QUERY_LABELS = [CLASS_NAMES[int(embedding_labels[index])] for index in QUERY_INDICES]

print("Query indices:", QUERY_INDICES)
print("Query classes:", QUERY_LABELS)
assert len(set(QUERY_LABELS)) == 3
log(f"Nearest-neighbor query_indices={QUERY_INDICES} query_classes={QUERY_LABELS}")
print("PART D EMBEDDINGS: PASS")


In [ ]:
def retrieve_neighbors(embeddings, query_index, top_k=5):
    similarities = embeddings @ embeddings[query_index]
    similarities = similarities.clone()
    similarities[query_index] = float("-inf")
    values, indices = torch.topk(similarities, k=top_k)
    return indices.tolist(), values.tolist()

encoder_embeddings = {
    "Supervised": supervised_embeddings,
    "Rotation SSL": rotation_embeddings,
    "SimCLR": simclr_embeddings,
}

neighbor_rows = []

for query_index in QUERY_INDICES:
    query_image, query_label = test_raw[query_index]
    fig, axes = plt.subplots(3, 6, figsize=(18, 9))

    for row_index, (encoder_name, embeddings) in enumerate(encoder_embeddings.items()):
        neighbor_indices, similarities = retrieve_neighbors(
            embeddings, query_index, top_k=5
        )

        axes[row_index, 0].imshow(query_image)
        axes[row_index, 0].set_title(
            f"{encoder_name}\nQUERY {query_index}\n{CLASS_NAMES[int(query_label)]}"
        )
        axes[row_index, 0].axis("off")

        for rank, (neighbor_index, similarity) in enumerate(
            zip(neighbor_indices, similarities), start=1
        ):
            neighbor_image, neighbor_label = test_raw[neighbor_index]
            axes[row_index, rank].imshow(neighbor_image)
            axes[row_index, rank].set_title(
                f"Top {rank}: {CLASS_NAMES[int(neighbor_label)]}\n"
                f"idx={neighbor_index}, cos={similarity:.3f}"
            )
            axes[row_index, rank].axis("off")

            neighbor_rows.append({
                "encoder": encoder_name,
                "query_index": query_index,
                "query_label": CLASS_NAMES[int(query_label)],
                "neighbor_rank": rank,
                "neighbor_index": neighbor_index,
                "neighbor_label": CLASS_NAMES[int(neighbor_label)],
                "cosine_similarity": similarity,
                "label_match": int(neighbor_label == query_label),
            })

    fig.suptitle(
        f"Top-5 cosine neighbors for shared query {query_index} "
        f"({CLASS_NAMES[int(query_label)]})",
        fontsize=16,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    figure_path = FIGURE_DIR / f"nearest_neighbors_query_{query_index}.png"
    plt.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.show()

neighbors_df = pd.DataFrame(neighbor_rows)
neighbors_df.to_csv(OUTPUT_DIR / "nearest_neighbors.csv", index=False)

neighbor_purity = (
    neighbors_df.groupby("encoder")["label_match"].mean().sort_values(ascending=False)
)
purity_table = (100 * neighbor_purity).rename("top5_label_purity_percent").reset_index()
display(purity_table.round(2))

print("Each encoder produced", len(QUERY_INDICES) * 5, "retrieved neighbors.")
print("PART D: PASS")


# Final Results, Comparison, and Required Discussion


In [ ]:
metrics_df = pd.DataFrame([
    {
        "method": "Supervised (500 labels)",
        "backbone": "ResNet-18, weights=None",
        "pretraining_images": 500,
        "pretraining_epochs": SUPERVISED_EPOCHS,
        "linear_probe_epochs": 0,
        "test_accuracy_percent": 100 * part_a_test["accuracy"],
        "test_loss": part_a_test["loss"],
        "top5_neighbor_purity_percent": 100 * neighbor_purity["Supervised"],
    },
    {
        "method": "Rotation SSL + linear probe",
        "backbone": "ResNet-18, weights=None",
        "pretraining_images": len(rotation_dataset),
        "pretraining_epochs": ROTATION_EPOCHS,
        "linear_probe_epochs": ROTATION_LINEAR_EPOCHS,
        "test_accuracy_percent": 100 * part_b_test["accuracy"],
        "test_loss": part_b_test["loss"],
        "top5_neighbor_purity_percent": 100 * neighbor_purity["Rotation SSL"],
    },
    {
        "method": "SimCLR + linear probe",
        "backbone": "ResNet-18, weights=None",
        "pretraining_images": len(simclr_dataset),
        "pretraining_epochs": SIMCLR_EPOCHS,
        "linear_probe_epochs": SIMCLR_LINEAR_EPOCHS,
        "test_accuracy_percent": 100 * part_c_test["accuracy"],
        "test_loss": part_c_test["loss"],
        "top5_neighbor_purity_percent": 100 * neighbor_purity["SimCLR"],
    },
])

display(metrics_df.round(3))
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

best_row = metrics_df.sort_values("test_accuracy_percent", ascending=False).iloc[0]
best_purity_row = metrics_df.sort_values(
    "top5_neighbor_purity_percent", ascending=False
).iloc[0]

analysis_paragraphs = [
    (
        f"The strongest test classifier was {best_row['method']} with "
        f"{best_row['test_accuracy_percent']:.2f}% accuracy. All three methods used the "
        "same unpretrained ResNet-18 architecture and the same balanced 500-image labeled "
        "subset, so the main experimental difference was how each representation was learned."
    ),
    (
        f"The supervised baseline achieved {metrics_df.iloc[0]['test_accuracy_percent']:.2f}% "
        "accuracy. It received direct class supervision but only from 500 images, so it can "
        "learn task-specific boundaries while also being vulnerable to overfitting. Stronger "
        "regularization, tuned augmentation, longer scheduling, or additional labels could improve it."
    ),
    (
        f"Rotation SSL achieved {metrics_df.iloc[1]['test_accuracy_percent']:.2f}% accuracy "
        f"after learning from all {len(rotation_dataset):,} unlabeled images. Rotation prediction "
        "encourages shape and orientation features, but the proxy task does not always align with "
        "semantic object categories. Longer pretraining, multi-task pretexts, or carefully tuned "
        "spatial augmentations could improve this representation."
    ),
    (
        f"SimCLR achieved {metrics_df.iloc[2]['test_accuracy_percent']:.2f}% accuracy after "
        f"contrastive training on {len(simclr_dataset):,} unlabeled images. Its result depends on "
        "augmentation quality, batch negatives, training duration, and temperature. A larger batch, "
        "more pretraining epochs, a learning-rate warmup, or a larger unlabeled subset could improve it."
    ),
    (
        f"The nearest-neighbor visualizations used identical queries and cosine similarity for all "
        f"encoders. {best_purity_row['method']} had the highest top-5 label purity at "
        f"{best_purity_row['top5_neighbor_purity_percent']:.2f}%. Rows containing visually and "
        "semantically consistent neighbors indicate a representation that clusters class-relevant "
        "features; mismatched neighbors reveal sensitivity to background, color, pose, or low-level texture."
    ),
]

print("\nREQUIRED COMPARISON\n")
for paragraph in analysis_paragraphs:
    print(paragraph, "\n")
    log("Analysis: " + paragraph)

history_df = pd.DataFrame(
    supervised_history
    + rotation_history
    + rotation_linear_history
    + simclr_history
    + simclr_linear_history
)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for stage, group in history_df.groupby("stage"):
    axes[0].plot(group["epoch"], group["loss"], marker="o", label=stage)
axes[0].set_title("Training Loss by Stage")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].bar(
    metrics_df["method"],
    metrics_df["test_accuracy_percent"],
    color=["#4c78a8", "#f58518", "#54a24b"],
)
axes[1].set_title("STL-10 Test Accuracy")
axes[1].set_ylabel("Accuracy (%)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.25)
for index, value in enumerate(metrics_df["test_accuracy_percent"]):
    axes[1].text(index, value + 0.5, f"{value:.2f}%", ha="center")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "training_and_accuracy_summary.png", dpi=180, bbox_inches="tight")
plt.show()

print("FINAL COMPARISON: PASS")


## Create final submission artifacts


In [ ]:
def half_precision_state_dict(module):
    compact = {}
    for key, value in module.state_dict().items():
        tensor = value.detach().cpu()
        compact[key] = tensor.half() if tensor.is_floating_point() else tensor
    return compact

def save_compact_model(path, encoder, classifier, method):
    payload = {
        "method": method,
        "architecture": "torchvision.models.resnet18(weights=None)",
        "encoder_state_dict_fp16": half_precision_state_dict(encoder),
        "classifier_state_dict_fp16": half_precision_state_dict(classifier),
        "class_names": CLASS_NAMES,
        "image_size": IMAGE_SIZE,
        "normalization_mean": STL10_MEAN,
        "normalization_std": STL10_STD,
        "seed": SEED,
    }
    torch.save(payload, path)
    print(f"Saved {path.name}: {path.stat().st_size / 1024**2:.2f} MiB")

save_compact_model(
    MODEL_DIR / "part_a_supervised_resnet18_fp16.pt",
    supervised_model.encoder,
    supervised_model.classifier,
    "Part A supervised",
)
save_compact_model(
    MODEL_DIR / "part_b_rotation_ssl_resnet18_fp16.pt",
    rotation_linear_model.encoder,
    rotation_linear_model.classifier,
    "Part B rotation SSL plus linear probe",
)
save_compact_model(
    MODEL_DIR / "part_c_simclr_resnet18_fp16.pt",
    simclr_linear_model.encoder,
    simclr_linear_model.classifier,
    "Part C SimCLR plus linear probe",
)

run_config = {
    "student": "Nikhil Kanaparthi",
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
    "dataset": "STL-10",
    "backbone": "ResNet-18, weights=None",
    "image_size": IMAGE_SIZE,
    "labeled_subset_size": LABELED_SUBSET_SIZE,
    "labeled_sampling": "balanced, 50 images per class",
    "rotation_unlabeled_size": len(rotation_dataset),
    "simclr_unlabeled_size": len(simclr_dataset),
    "simclr_temperature": TEMPERATURE,
    "epochs": {
        "supervised": SUPERVISED_EPOCHS,
        "rotation_ssl": ROTATION_EPOCHS,
        "rotation_linear": ROTATION_LINEAR_EPOCHS,
        "simclr": SIMCLR_EPOCHS,
        "simclr_linear": SIMCLR_LINEAR_EPOCHS,
    },
    "query_indices": QUERY_INDICES,
}
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_memory_gib": GPU_MEMORY_GIB,
}
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

analysis_md = "# Homework 6 Analysis\n\n" + "\n\n".join(analysis_paragraphs) + "\n"
(OUTPUT_DIR / "ANALYSIS.md").write_text(analysis_md, encoding="utf-8")

metrics_markdown = f"""# Homework 6 Metrics

Student: Nikhil Kanaparthi  
SID4: 6441  
Seed: 6441  
Dataset: STL-10  
Backbone: ResNet-18 with no pretrained weights

## Model comparison

{metrics_df.round(4).to_markdown(index=False)}

## Nearest-neighbor label purity

{purity_table.round(3).to_markdown(index=False)}

## Required analysis

{chr(10).join(analysis_paragraphs)}
"""
(OUTPUT_DIR / "METRICS.md").write_text(metrics_markdown, encoding="utf-8")

ai_use_md = """# AI Use - Homework 6

I used an AI assistant to develop a first-cut notebook structure, adapt the provided self-supervised learning demo from CIFAR-10 to STL-10, and implement the assignment requirements with a common ResNet-18 backbone. The assistance helped me understand the differences between end-to-end supervision, rotation pretraining, SimCLR contrastive learning, frozen linear evaluation, cosine nearest-neighbor retrieval, and checkpoint-based recovery in Colab.

I ran the notebook incrementally, reviewed the code and outputs, verified that the labeled subset contained exactly 500 balanced examples, confirmed that the rotation encoder used all 100,000 unlabeled images, and checked that SimCLR used two augmented views from 20,000 unlabeled images with temperature 0.2. I also confirmed that only each linear classification head was trainable during the frozen-encoder evaluations and based the final comparison on the measured accuracies and nearest-neighbor results.
"""
(OUTPUT_DIR / "AI_USE.md").write_text(ai_use_md, encoding="utf-8")

readme_md = f"""# DATA 266 Homework 6 - STL-10 Representation Learning

Student: Nikhil Kanaparthi  
SID4: 6441

## Overview

This submission compares three ResNet-18 representations trained from scratch on STL-10:

1. End-to-end supervised learning with 500 balanced labeled images.
2. Rotation self-supervised learning on all 100,000 unlabeled images plus a frozen linear probe.
3. SimCLR-style contrastive learning on 20,000 unlabeled images plus a frozen linear probe.

## Main files

- `HW6_STL10_SSL_Nikhil_Kanaparthi_6441.ipynb`: executed notebook; add after downloading it from Colab.
- `METRICS.md`: accuracy and nearest-neighbor summary.
- `ANALYSIS.md`: required comparison and interpretation.
- `AI_USE.md`: AI-use disclosure.
- `RUN_LOG.txt`: timestamped execution record.
- `metrics.csv`: model-level results.
- `training_history.csv`: epoch-level histories.
- `nearest_neighbors.csv`: top-5 cosine retrieval records.
- `figures/`: training plots, augmentations, and nearest-neighbor visualizations.
- `models/`: compact half-precision archival state dictionaries for the three final models.
- `run_config.json` and `environment.json`: reproducibility metadata.

## Important controls

- All encoders use `torchvision.models.resnet18(weights=None)`.
- The same seeded, balanced 500-image subset is used for all classifier training.
- Rotation SSL uses all 100,000 unlabeled STL-10 images.
- SimCLR uses 20,000 unlabeled images, two views, cosine similarity, and temperature 0.2.
- Rotation and SimCLR encoders remain frozen during linear evaluation.
- The same query images are used for all nearest-neighbor comparisons.

## Final measured result

The best test method in this run was **{best_row['method']}** with **{best_row['test_accuracy_percent']:.2f}%** accuracy.
"""
(OUTPUT_DIR / "README.md").write_text(readme_md, encoding="utf-8")

log(f"Saved final artifacts to {OUTPUT_DIR}")
(OUTPUT_DIR / "RUN_LOG.txt").write_text(
    "\n".join(RUN_LOG_LINES) + "\n", encoding="utf-8"
)

manifest_lines = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        manifest_lines.append(
            f"{path.relative_to(OUTPUT_DIR)}\t{path.stat().st_size} bytes"
        )
(OUTPUT_DIR / "submission_manifest.txt").write_text(
    "\n".join(manifest_lines) + "\n", encoding="utf-8"
)

print("\nCreated submission files:")
for line in manifest_lines:
    print(" -", line)
print("ARTIFACT GENERATION: PASS")


## Package and download the runtime artifacts


In [ ]:
archive_base = WORK_ROOT / "HW6_STL10_SSL_Nikhil_Kanaparthi_6441_runtime_artifacts"
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=WORK_ROOT,
        base_dir=OUTPUT_DIR.name,
    )
)

print("Created:", archive_path)
print(f"Archive size: {archive_path.stat().st_size / 1024**2:.2f} MiB")

if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    print("Download the ZIP from:", archive_path)

print("PACKAGING: PASS")


# Final submission checklist and recommended GitHub commits

Before submitting:

- [ ] `FAST_DEV_RUN` was `False`.
- [ ] All cells were executed and no red traceback remains.
- [ ] Part A used exactly 500 balanced labels and ran 15 epochs.
- [ ] Rotation SSL used all 100,000 unlabeled images and ran 15 epochs.
- [ ] Both frozen linear probes ran 20 epochs with zero trainable encoder parameters.
- [ ] SimCLR used exactly 20,000 unlabeled images, the four demo augmentations, and temperature 0.2.
- [ ] Three different queries each show five neighbors for all three encoders.
- [ ] The executed notebook was downloaded separately and copied into the extracted artifact folder.
- [ ] `AI_USE.md`, `METRICS.md`, `ANALYSIS.md`, `RUN_LOG.txt`, figures, CSV files, and model files are present.
- [ ] GitHub tag `hw6` points to the final commit.

Recommended sequential commits:

1. `HW6: add STL-10 notebook structure and balanced subset`
2. `HW6: complete limited-label supervised baseline`
3. `HW6: complete rotation SSL and frozen linear probe`
4. `HW6: complete SimCLR pretraining and linear evaluation`
5. `HW6: add cosine nearest-neighbor comparison`
6. `HW6: add final metrics analysis models and documentation`

Submit the tagged GitHub URL through Canvas. A separate report PDF is not requested; the executed notebook, `ANALYSIS.md`, and `METRICS.md` provide the report content.
